[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KUMARAGURU-V-S/Lab-experiments/blob/main/GEN-AI-AND-LLM/Experiment-10-Fine-Tuning-PreTrained-Model/finetune_lm.ipynb)

# finetune_lm.py

In [1]:
!pip install -q -U transformers datasets accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.8 MB/s eta 0:00:00


In [2]:
# Experiment 10: Fine-Tuning a Pre-Trained Language Model
# Domain-Specific Application: IMDB Sentiment Classification

import numpy as np
from datasets import load_dataset
from sklearn.metrics import accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

# ---------------------------------------------------------
# 1. Load Dataset
# ---------------------------------------------------------

print("Loading IMDB movie reviews dataset...")

dataset = load_dataset("stanfordnlp/imdb")

print(dataset)

# Use a smaller subset so that it runs faster in Colab
train_dataset = (
    dataset["train"]
    .shuffle(seed=42)
    .select(range(2000))
)

test_dataset = (
    dataset["test"]
    .shuffle(seed=42)
    .select(range(500))
)

print("\nTraining samples:", len(train_dataset))
print("Testing samples :", len(test_dataset))


# ---------------------------------------------------------
# 2. Load Pre-trained Tokenizer
# ---------------------------------------------------------

print("\nLoading DistilBERT tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)


# ---------------------------------------------------------
# 3. Tokenize Dataset
# ---------------------------------------------------------

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )


print("Tokenizing datasets...")

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)

# Remove original text column
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

# Rename label to labels for Transformers
train_dataset = train_dataset.rename_column(
    "label", "labels"
)

test_dataset = test_dataset.rename_column(
    "label", "labels"
)

# Set PyTorch format
train_dataset.set_format("torch")
test_dataset.set_format("torch")

print("Tokenization completed.")


# ---------------------------------------------------------
# 4. Load Pre-trained DistilBERT Model
# ---------------------------------------------------------

print("\nLoading pre-trained DistilBERT model...")

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

print("Model loaded successfully.")


# ---------------------------------------------------------
# 5. Define Evaluation Metric
# ---------------------------------------------------------

def compute_metrics(eval_prediction):

    predictions = eval_prediction.predictions
    labels = eval_prediction.label_ids

    predicted_labels = np.argmax(
        predictions,
        axis=1
    )

    accuracy = accuracy_score(
        labels,
        predicted_labels
    )

    return {
        "accuracy": accuracy
    }


# ---------------------------------------------------------
# 6. Training Configuration
# ---------------------------------------------------------

training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=2,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=5e-5,

    logging_steps=50,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    report_to="none"
)


# ---------------------------------------------------------
# 7. Create Trainer
# ---------------------------------------------------------

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)


# ---------------------------------------------------------
# 8. Fine-Tune Model
# ---------------------------------------------------------

print("\n========================================")
print("Starting fine-tuning...")
print("========================================\n")

trainer.train()


# ---------------------------------------------------------
# 9. Evaluate Fine-Tuned Model
# ---------------------------------------------------------

print("\n========================================")
print("Evaluating fine-tuned model...")
print("========================================\n")

evaluation_results = trainer.evaluate()

print("\nEvaluation Results:")

for key, value in evaluation_results.items():
    print(f"{key}: {value}")


# ---------------------------------------------------------
# 10. Save Fine-Tuned Model
# ---------------------------------------------------------

save_directory = "./fine_tuned_distilbert_imdb"

trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

print("\n========================================")
print("Model saved successfully!")
print("Location:", save_directory)
print("========================================")

Loading IMDB movie reviews dataset...


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
0.229173,0.462233,2,0.808000



Evaluation Results:
eval_loss: 0.4622334837913513
eval_accuracy: 0.808


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved successfully!
Location: ./fine_tuned_distilbert_imdb
